In [ ]:
# ===============================
# Telugu Text Normalization – IndicBART NaN-SAFE FINAL
# ===============================

!pip install -q transformers datasets sentencepiece accelerate sacrebleu indic-nlp-library

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# -------------------------------
# Load & clean dataset
# -------------------------------
df = pd.read_csv(
    "telugu_normalization_Dataset_Pairs.tsv",
    sep="\t",
    header=None,
    names=["noisy", "clean"],
    keep_default_na=False,
    dtype=str
)

def strip_invisibles(s):
    return (
        s.replace("\u200c", "")
         .replace("\u200d", "")
         .replace("\ufeff", "")
         .strip()
    )

df["noisy"] = df["noisy"].apply(strip_invisibles)
df["clean"] = df["clean"].apply(strip_invisibles)

# HARD SAFETY FILTERS (critical)
df = df[df["clean"].str.len() >= 2]
df = df[df["clean"].str.split().str.len() >= 2]

df = df.reset_index(drop=True)
print("Final rows:", len(df))

# -------------------------------
# Train / Val split
# -------------------------------
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

# -------------------------------
# Model & tokenizer
# -------------------------------
MODEL_NAME = "ai4bharat/indicbart"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ===== CORRECT IndicBART CONFIG =====
model.config.decoder_start_token_id = tokenizer.bos_token_id

model = model.cuda()

# -------------------------------
# Preprocessing (NaN-proof)
# -------------------------------
MAX_LEN = 96

def preprocess(batch):
    inputs = tokenizer(
        batch["noisy"],
        max_length=MAX_LEN,
        truncation=True,
        padding=False
    )

    targets = tokenizer(
        batch["clean"],
        max_length=MAX_LEN,
        truncation=True,
        padding=False
    )

    labels = []
    for seq in targets["input_ids"]:
        labels.append([
            -100 if tok == tokenizer.pad_token_id else tok
            for tok in seq
        ])

    inputs["labels"] = labels
    return inputs

# -------------------------------
# Dataset
# -------------------------------
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

# -------------------------------
# Training arguments
# -------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./indicbart_telugu_norm_fixed",
    eval_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    warmup_steps=500,
    num_train_epochs=8,
    fp16=False,   # turn ON after confirming no NaNs
    logging_steps=200,
    save_steps=1000,
    eval_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

# -------------------------------
# Trainer
# -------------------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

# -------------------------------
# Train
# -------------------------------
trainer.train()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 6.7 MB/s eta 0:00:00
Final rows: 41093


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

...

Map:   0%|          | 0/4110 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


Step,Training Loss,Validation Loss
1000,0.922600,0.689322
2000,0.535600,0.429168
3000,0.410000,0.338744
4000,0.344900,0.304408
5000,0.305000,0.273674
6000,0.288600,0.248873
7000,0.263900,0.236525
8000,0.252300,0.221462
9000,0.238900,0.215477
10000,0.232300,0.202717


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=18496, training_loss=0.3774167954096745, metrics={'train_runtime': 13425.6884, 'train_samples_per_second': 22.037, 'train_steps_per_second': 1.378, 'total_flos': 2.263221098378035e+16, 'train_loss': 0.3774167954096745, 'epoch': 8.0})

In [ ]:
!zip -r indicbart_telugu_norm_checkpoint.zip indicbart_telugu_norm_fixed/checkpoint-18496


  adding: indicbart_telugu_norm_fixed/checkpoint-18496/ (stored 0%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/trainer_state.json (deflated 77%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/generation_config.json (deflated 34%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/scheduler.pt (deflated 62%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/training_args.bin (deflated 53%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/tokenizer_config.json (deflated 87%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/rng_state.pth (deflated 26%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/optimizer.pt (deflated 8%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/spiece.model (deflated 61%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/added_tokens.json (deflated 61%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/special_tokens_map.json (deflated 58%)
  adding: indicbart_telugu_norm_fixed/checkpoint-18496/m

In [ ]:
!zip -r indicbart_tokenizer.zip ~/.cache/huggingface/hub/models--ai4bharat--indicbart


  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/ (stored 0%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/ (stored 0%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/78466a0c0e29f9229f7005623ecd6bc4243c0ae0/ (stored 0%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/78466a0c0e29f9229f7005623ecd6bc4243c0ae0/tokenizer_config.json (deflated 50%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/78466a0c0e29f9229f7005623ecd6bc4243c0ae0/spiece.model (deflated 61%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/78466a0c0e29f9229f7005623ecd6bc4243c0ae0/added_tokens.json (deflated 58%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/78466a0c0e29f9229f7005623ecd6bc4243c0ae0/pytorch_model.bin (deflated 7%)
  adding: root/.cache/huggingface/hub/models--ai4bharat--indicbart/snapshots/78466a0c0e29f9229f70

In [ ]:
from google.colab import files
files.download("indicbart_telugu_norm_checkpoint.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("indicbart_tokenizer.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#---------------------Model Testing-------------------

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

ckpt_path = "indicbart_telugu_norm_fixed/checkpoint-18496"  # adjust if needed

tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/indicbart",
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(ckpt_path)
model = model.cuda()
model.eval()


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(64014, 1024, padding_idx=0)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(64014, 1024, padding_idx=0)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-5): 6 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
  

In [ ]:
def normalize_telugu(text, max_len=96):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )

    # 🚨 FIX: remove token_type_ids if present
    inputs.pop("token_type_ids", None)

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_len,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
tests = [
    "నేను పాఠశాలకు వెళ్తున్నాను",
    "ఇది ఒక మంచి రోజు",
    "భారతదేశం ఒక గొప్ప దేశం"
]

for t in tests:
    print("INPUT :", t)
    print("OUTPUT:", normalize_telugu(t))
    print("-" * 40)


INPUT : నేను పాఠశాలకు వెళ్తున్నాను
OUTPUT: నేను పాశాలకు వె్తున్నాను
----------------------------------------
INPUT : ఇది ఒక మంచి రోజు
OUTPUT: ఇది ఒక మంచి రోజు
----------------------------------------
INPUT : భారతదేశం ఒక గొప్ప దేశం
OUTPUT: ారతదేశం ఒక గొప్ప దేశం
----------------------------------------


In [ ]:
ocr_tests = [
    # ---- Single word cases (MOST IMPORTANT) ----
    "పాఠశాల",
    "భారతదేశం",
    "విద్యార్థి",
    "తెలుగు",
    "స్వాతంత్ర్యం",

    # ---- Space / merge errors ----
    "నేనుపాఠశాలకు వెళ్తున్నాను",
    "ఇదిఒకమంచిరోజు",
    "భారతదేశంఒకగొప్పదేశం",
    "పరిశీలనకోసం వాక్యం",

    # ---- Extra / missing spaces ----
    "నేను  పాఠశాలకు  వెళ్తున్నాను",
    "ఇది ఒకమంచి రోజు",
    "భారత దేశం ఒక గొప్పదేశం",

    # ---- OCR glyph confusion ----
    "నేను పాఠశాలకు వె్తున్నాను",
    "భారతదేశం ఒక గొప్ప దశం",
    "విద్యార్తి పుస్తకం చదువుతున్నాడు",

    # ---- Edge / short ----
    "అ",
    "నేను",
    "దేశం"
]


In [ ]:
for t in ocr_tests:
    out = normalize_telugu(t)
    print("INPUT :", t)
    print("OUTPUT:", out)
    print("-" * 50)


INPUT : పాఠశాల
OUTPUT: పాశాల
--------------------------------------------------
INPUT : భారతదేశం
OUTPUT: ారతదేశం
--------------------------------------------------
INPUT : విద్యార్థి
OUTPUT: విద్యార్ి
--------------------------------------------------
INPUT : తెలుగు
OUTPUT: తెలుగు
--------------------------------------------------
INPUT : స్వాతంత్ర్యం
OUTPUT: స్వాతంత్ర్యం
--------------------------------------------------
INPUT : నేనుపాఠశాలకు వెళ్తున్నాను
OUTPUT: నేనుపాశాలకు వె్తున్నాను
--------------------------------------------------
INPUT : ఇదిఒకమంచిరోజు
OUTPUT: ఇది ఒక మంచిరోజు
--------------------------------------------------
INPUT : భారతదేశంఒకగొప్పదేశం
OUTPUT: ారతదేశం ఒకగొప్పదేశం
--------------------------------------------------
INPUT : పరిశీలనకోసం వాక్యం
OUTPUT: పరిశీలనకోసం వాక్యం
--------------------------------------------------
INPUT : నేను  పాఠశాలకు  వెళ్తున్నాను
OUTPUT: నేను పాశాలకు వె్తున్నాను
--------------------------------------------------
INPUT : ఇది ఒకమంచి రోజు
OUT